## Welcome to CS 229 Notebook

## Computing Speeeeed per the Tang paper

### 1a. Download Kaggle and UCF 101


In [ ]:
from google.colab import drive
from pathlib import Path
import os, shutil, kagglehub

# mount file system
drive.mount('/content/drive')
DRIVE_ROOT = Path("/content/drive/MyDrive/cs229") # set as root
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)

local_root = Path(kagglehub.dataset_download("pevogam/ucf101")) # kaggle root
print("kagglehub local_root:", local_root)

# move from Kaggle to drive by looking for all .avi files
# store it locally before downloading it to drive
def find_ucf_root(base: Path):
    for root, dirs, files in os.walk(base):
        if any(f.endswith(".avi") for f in files):
            return Path(root).parent # stored locally before we are movin the file
    return None

ucf_local_root = find_ucf_root(local_root)


# copy the local root to Drive root.
DRIVE_UCF_ROOT = DRIVE_ROOT / "UCF-101"
print("Copying to:", DRIVE_UCF_ROOT)
shutil.copytree(ucf_local_root, DRIVE_UCF_ROOT, dirs_exist_ok=True) # copy the files over

# Use this everywhere else:
UCF_ROOT = DRIVE_UCF_ROOT # set UCF root for later usage
print("UCF_ROOT =", UCF_ROOT) # make sure this is UCF-101 root

Mounted at /content/drive
kagglehub local_root: /root/.cache/kagglehub/datasets/pevogam/ucf101/versions/1
Detected UCF local root: /root/.cache/kagglehub/datasets/pevogam/ucf101/versions/1/UCF101/UCF-101
Copying to: /content/drive/MyDrive/cs229/UCF-101
Done. Example classes: ['SalsaSpin', 'BabyCrawling', 'BodyWeightSquats', 'PlayingFlute', 'Skiing', 'Archery', 'HeadMassage', 'FrontCrawl', 'JumpingJack', 'JavelinThrow']
UCF_ROOT = /content/drive/MyDrive/cs229/UCF-101


### 1b. Set up storage for keypoints


In [ ]:
!pip install -q datasets transformers accelerate
UCF_ROOT = DRIVE_ROOT / "UCF-101" # videos (already copied)
POSE_OUT = DRIVE_ROOT / "vitpose_sub8_train_npz" # we will save keypoints here
POSE_OUT.mkdir(parents=True, exist_ok=True)
print("UCF_ROOT:", UCF_ROOT, "exists:", UCF_ROOT.exists()) # checkkc
print("POSE_OUT:", POSE_OUT) # check

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
UCF_ROOT: /content/drive/MyDrive/cs229/UCF-101 exists: True
POSE_OUT: /content/drive/MyDrive/cs229/vitpose_sub8_train_npz


### 1c. Running Pose Detection up to here all is good

In [ ]:
# Install once per runtime (initializing the larger process)
# Code is taken from the HF usage of ViTPose
%pip install -q opencv-python supervision transformers torch

import torch
import numpy as np
import cv2

from transformers import (
    AutoProcessor,
    RTDetrForObjectDetection,
    VitPoseForPoseEstimation,
    infer_device,
)

device = infer_device()
print("Using device:", device) # GPU A100 accelerated on colab

# First do RT-DETR
det_processor = AutoProcessor.from_pretrained("PekingU/rtdetr_r50vd_coco_o365")
det_model = RTDetrForObjectDetection.from_pretrained(
    "PekingU/rtdetr_r50vd_coco_o365"
).to(device)
det_model.eval()

# Then run ViTPose
pose_processor = AutoProcessor.from_pretrained("usyd-community/vitpose-base-simple")
pose_model = VitPoseForPoseEstimation.from_pretrained(
    "usyd-community/vitpose-base-simple"
).to(device)
pose_model.eval()

Using device: cuda


In [ ]:
def detect_person_bbox(frame_rgb: np.ndarray):
"""
Run RT-DETR on a single RGB frame.
Returns (x1, y1, x2, y2) for highest-scoring 'person'
THIS IS THE CONFIDENCE GATE.
"""
  h, w, _ = frame_rgb.shapeinputs = det_processor(images=frame_rgb, return_tensors="pt").to(device)
  with torch.no_grad():
    outputs = det_model(**inputs)

  results = det_processor.post_process_object_detection(outputs, threshold=0.5, target_sizes=[(h, w)])[0]
  # store boxes to select high confidence
  boxes  = results["boxes"]
  boxes  = boxes[mask]

  # store scores and labels
  scores = results["scores"]
  labels = results["labels"]
  mask = labels == PERSON_LABEL_ID

  if mask.sum() == 0:
    return None # edge case

  scores = scores[mask]
  best = scores.argmax().item() # FIRST CONFIDENCE GATE
  x1, y1, x2, y2 = boxes[best].tolist() # guess for motion of individual location for ViTPose
  x1 = max(0, int(x1)); y1 = max(0, int(y1))
  x2 = min(w, int(x2)); y2 = min(h, int(y2))
  if x2 <= x1 or y2 <= y1:
      return None
  return x1, y1, x2, y2


def run_vitpose_on_frame(frame_rgb: np.ndarray, bbox):
    """
    frame_rgb: (H, W, 3) RGB
    bbox: (x1, y1, x2, y2) in full-frame coords
    Returns keypoints (K, 3) in full-frame coords (x, y, score) or None.
    """
    x1, y1, x2, y2 = bbox
    w = x2 - x1
    h = y2 - y1
    if w <= 0 or h <= 0:
        return None

    # ViTPose expects COCO [x, y, w, h] boxes
    person_boxes = np.array([[x1, y1, w, h]], dtype=np.float32)

    inputs = pose_processor(
        frame_rgb,
        boxes=[person_boxes],       # list of boxes for this image
        return_tensors="pt",
    ).to(device)

    with torch.no_grad():
        outputs = pose_model(**inputs)

    pose_results = pose_processor.post_process_pose_estimation(
        outputs,
        boxes=[person_boxes],
        threshold=0.3,
    )

    result = pose_results[0]
    if len(result) == 0:
        return None

    keypoints = result[0]["keypoints"]  # (K, 3), already in full-frame coords
    return keypoints.cpu().numpy().astype(np.float32)

In [ ]:
import numpy as np
"""
For weak labeling, we pick to sample 1 in every 8th frame
The reason we selected 8 was because estimated ~2x the length of the video for
iterate runtime. 1/8 we estimated would take ~ 1/4 length so we just had to run
for ~7hrs (27/4...)
"""
def sample_frame_indices(total_frames: int, stride: int = 8):
"""
Return list of frame indices to process
"""
  return list(range(0, total_frames, stride))


In [ ]:
from tqdm import tqdm

processed = 0
skipped   = 0

for video_path, clip_id in tqdm(iter_train_videos(UCF_ROOT, train_clip_set),
                                desc="ViTPose on train clips"):
  out_path = POSE_OUT / f"{clip_id}.npz" # store poses and then

  # Skip if already done (useful if you resume after disconnect)
  if out_path.exists():
      processed += 1
      continue

  cap = cv2.VideoCapture(str(video_path))
  if not cap.isOpened():
      print("[WARN] Cannot open:", video_path)
      skipped += 1
      continue

  total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT)) # no frames skip
  if total_frames <= 0:
    skipped += 1
    cap.release()
    continue

  frame_indices = sample_frame_indices(total_frames, stride=8)
  keypoints_list = []

  for idx in frame_indices:
    cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
    ret, frame_bgr = cap.read()
    if not ret:
      break

    frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)

    # 1) person detection (RT-DETR)
    bbox = detect_person_bbox(frame_rgb)
    if bbox is None:
    # no person = store NaNs for frame
      keypoints_list.append(np.full((NUM_KEYPOINTS, 3), np.nan, dtype=np.float32))
      continue

    # 2) pose estimation (ViTPose) on full frame + bbox
    raw_kpts = run_vitpose_on_frame(frame_rgb, bbox)

    # Start with all-NaN array
    clean_kpts = np.full((NUM_KEYPOINTS, 3), np.nan, dtype=np.float32)

    if raw_kpts is not None:
      arr = np.asarray(raw_kpts, dtype=np.float32)

    # Expect something like (K, 3) so if do some shit  (K, 2)
    if arr.ndim == 2 and arr.shape[0] > 0:
    # If only x,y are present, add a dummy confidence column
      if arr.shape[1] == 2:
        scores = np.ones((arr.shape[0], 1), dtype=np.float32)
        arr = np.concatenate([arr, scores], axis=1)

# YANAV I AM SO FICKING TIRJFB PBOR J+OJK *BOUV"+LQKöKEöHIDQKV<dc

        # Now arr should be (K, >=3); take first 3 cols
        arr = arr[:, :3]

        # Truncate or pad to NUM_KEYPOINTS
        K = min(NUM_KEYPOINTS, arr.shape[0])
        clean_kpts[:K, :] = arr[:K, :]
# fucucu243ubewjrbkenr fuck meeee

      keypoints_list.append(clean_kpts)
    cap.release()

    if len(keypoints_list) == 0: # count for sskips
      skipped += 1
      continue

    keypoints_arr = np.stack(keypoints_list, axis=0)  # (T_sub, 17, 3) making sure size 3

    # SAVE DIRECTLY TO DRIVE
    np.savez_compressed(
        out_path,
        keypoints=keypoints_arr,
        clip_id=clip_id,
        video_path=str(video_path),
    )
    processed += 1 # ADD bc duh!
print("Done.")
print("Processed videos:", processed)
print("Skipped videos:", skipped)
print("Files saved in:", POSE_OUT)

ViTPose on train clips: 9537it [4:30:28,  1.70s/it]

Done.
Processed videos: 9537
Skipped videos: 0
Files saved in: /content/drive/MyDrive/cs229/vitpose_sub8_train_npz


## SPEED UPDATE = GOOD

In [ ]:
# Mount once
from google.colab import drive
drive.mount('/content/drive')

import numpy as np
from pathlib import Path
from joblib import Parallel, delayed
from tqdm import tqdm

# read raw ViTPose outputs straight off Drive
RAW_DIR = Path("/content/drive/MyDrive/cs229/vitpose_sub8_train_npz")

# single summary file we care about
JOINT_SPEEDS_OUT = Path("/content/drive/MyDrive/cs229/vitpose_sub8_train_joint_speeds.npz")

# GLOBAL VARs FOR PIPELINE (see paper)
# look team i set these nums bc i have been using them in lab, so trust they work.

CONF_THRESH = 0.8
MAX_ZERO_CONF_JOINTS = 8
EPS_ST = 8.0
NUM_KEYPOINTS = 17
ORIGINAL_FPS  = 25.0
FRAME_STRIDE  = 8

# COCO 17-keypoint indexing (from ViTPose)
NOSE = 0
L_EYE = 1
R_EYE = 2
L_EAR = 3
R_EAR = 4
L_SHOULDER = 5
R_SHOULDER = 6
L_ELBOW = 7
R_ELBOW = 8
L_WRIST = 9
R_WRIST = 10
L_HIP = 11
R_HIP = 12
L_KNEE = 13
R_KNEE = 14
L_ANKLE = 15
R_ANKLE = 16


def apply_confidence_gates(K_raw):
    """
    K_raw: (T,17,3) or (T,P,17,3)  (x,y,conf)
    Returns coords, conf with:
      - single best person
      - joints with conf < CONF_THRESH nuked to NaN / 0
      - frames with too many dead joints dropped
    """
    # Handle multi-person: pick person with highest mean conf
    if K_raw.ndim == 4:  # (T,P,17,3)
        conf_all = K_raw[..., 2]             # (T,P,17)
        mean_conf_per_person = np.nanmean(conf_all, axis=(0, 2))  # (P,)
        best_p = int(np.nanargmax(mean_conf_per_person))
        K = K_raw[:, best_p, :, :]           # (T,17,3)
    else:
        K = K_raw                             # (T,17,3)

    T = K.shape[0]
    if T == 0:
        return None

    coords = K[..., :2].copy()  # (T,17,2)
    conf   = K[..., 2].copy()   # (T,17)

    # drop low-conf joints: coords -> NaN, conf -> 0
    low_mask = conf < CONF_THRESH
    conf[low_mask] = 0.0
    coords[low_mask] = np.nan

    # drop frames with > MAX_ZERO_CONF_JOINTS dead joints
    zero_counts = (conf == 0.0).sum(axis=1)   # (T,)
    keep_frames = zero_counts <= MAX_ZERO_CONF_JOINTS
    if not keep_frames.any():
        return None  # nothing survives, rip

    coords = coords[keep_frames]  # (T',17,2)
    conf   = conf[keep_frames]    # (T',17)
    return coords, conf


def translation_and_scale_invariance(coords, conf):
    """
    coords: (T,17,2)
    conf:   (T,17)

    Hard-coded normalization:
      - ONLY accept frames where both shoulders are valid.
      - Origin = mid-shoulder.
      - Scale = shoulder distance.
      - If any of that fails, we THROW AWAY the frame.
    """
    T = coords.shape[0]
    coords_out = []
    conf_out   = []

    def available(xy_t, c_t, idx):
        # joint exists, has non-zero conf and finite coords
        return (c_t[idx] > 0) and (not np.any(np.isnan(xy_t[idx])))

    for t in range(T):
        xy = coords[t].copy()  # (17,2)
        c  = conf[t].copy()    # (17,)

        # all joints dead? nothing to normalize
        if np.all(c == 0):
            continue

        # we ONLY trust frames with BOTH shoulders
        if not (available(xy, c, L_SHOULDER) and available(xy, c, R_SHOULDER)):
            continue

        # origin = mid-shoulder
        origin = (xy[L_SHOULDER] + xy[R_SHOULDER]) / 2.0
        xy_shifted = xy - origin

        # scale = shoulder distance
        shoulder_vec = xy_shifted[L_SHOULDER] - xy_shifted[R_SHOULDER]
        shoulder_dist = float(np.linalg.norm(shoulder_vec))

        # if the body is collapsed into a pixel blob, skip
        if not np.isfinite(shoulder_dist) or (shoulder_dist < EPS_ST):
            continue

        # final normalized coords
        xy_norm = xy_shifted / shoulder_dist

        coords_out.append(xy_norm)
        conf_out.append(c)

    if len(coords_out) == 0:
        return None

    coords_out = np.stack(coords_out, axis=0)  # (T'',17,2)
    conf_out   = np.stack(conf_out,   axis=0)  # (T'',17)
    return coords_out, conf_out


def compute_joint_speed_17d(coords_norm, frame_stride=FRAME_STRIDE, base_fps=ORIGINAL_FPS):
    """
    coords_norm: (T,17,2) normalized joints with NaNs for missing.

    Vectorized velocity calc:
      - v_n(j) = (g_{n+1}(j) - g_{n-1}(j)) / (2Δt)
      - joint must be present at n-1 and n+1
      - s_n(j) = ||v_n(j)||_2
      - clip-level speed per joint = 95th percentile over n of s_n(j)
    """
    T = coords_norm.shape[0]
    if T < 3:
        # not enough frames for centered diff, so we just say “no motion”
        return np.zeros(NUM_KEYPOINTS, dtype=np.float32)

    dt = frame_stride / base_fps  # Δt in seconds

    # prev / next frames for centered difference, shape: (T-2, 17, 2)
    prev_xy = coords_norm[:-2]
    next_xy = coords_norm[2:]

    # joint must be present in BOTH frames
    valid_prev = ~np.isnan(prev_xy).any(axis=-1)   # (T-2,17)
    valid_next = ~np.isnan(next_xy).any(axis=-1)   # (T-2,17)
    valid = valid_prev & valid_next                # (T-2,17)

    # velocities and speeds for all frames/joints at once
    v = (next_xy - prev_xy) / (2.0 * dt)           # (T-2,17,2)
    speed = np.linalg.norm(v, axis=-1)             # (T-2,17)

    # nuke invalids to NaN so nanpercentile ignores them
    speed[~valid] = np.nan

    # 95th percentile per joint, ignoring NaNs
    with np.errstate(all="ignore"):
        S_joint = np.nanpercentile(speed, 95, axis=0)  # (17,)

    # joints that were never valid become NaN → set them to 0
    S_joint = np.where(np.isnan(S_joint), 0.0, S_joint).astype(np.float32)
    return S_joint  # (17,)


def process_one_npz(npz_path: Path):
    """
    Process a SINGLE clip:
      - load
      - confidence gating
      - shoulders-only normalization (anything else dies)
      - 17D speed vector

    Returns (clip_id, speed_17d) or None if clip dies in preprocessing.
    """
    try:
        data = np.load(npz_path, allow_pickle=True)
    except Exception:
        return None

    if "keypoints" not in data:
        return None

    K_raw = data["keypoints"]  # (T,17,3) or (T,P,17,3)

    gated = apply_confidence_gates(K_raw)
    if gated is None:
        return None
    coords, conf = gated  # (T',17,2), (T',17)

    normed = translation_and_scale_invariance(coords, conf)
    if normed is None:
        return None
    coords_norm, conf_norm = normed  # (T'',17,2), (T'',17)

    speed_17d = compute_joint_speed_17d(coords_norm)

    clip_id = data["clip_id"] if "clip_id" in data else npz_path.stem
    return clip_id, speed_17d

npz_files = sorted(RAW_DIR.glob("*.npz"))
print("Found raw files on Drive:", len(npz_files))

N_JOBS = 4

results = Parallel(n_jobs=N_JOBS, backend="loky", verbose=5)(
    delayed(process_one_npz)(p) for p in npz_files
)

# filter out the dead clips
clip_ids = []
speed_vecs = []
for r in results:
    if r is None:
        continue
    cid, v = r
    clip_ids.append(cid)
    speed_vecs.append(v.astype(np.float32))

if len(speed_vecs) == 0:
    raise RuntimeError("No clips survived preprocessing, something is off lol")

clip_ids_arr = np.array(clip_ids, dtype=object)
speed_17d_arr = np.stack(speed_vecs, axis=0)  # (N,17)

print("Final speed_17d_arr shape:", speed_17d_arr.shape)

# write ONE summary file back to Drive
np.savez_compressed(
    JOINT_SPEEDS_OUT,
    clip_ids=clip_ids_arr,
    speed_17d=speed_17d_arr,
)
print("Saved joint-speed summary to:", JOINT_SPEEDS_OUT)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Found raw files on Drive: 9537


[Parallel(n_jobs=4)]: Using backend LokyBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done  10 tasks      | elapsed:   45.9s
[Parallel(n_jobs=4)]: Done  64 tasks      | elapsed:  2.7min
[Parallel(n_jobs=4)]: Done 1232 tasks      | elapsed:  2.8min
[Parallel(n_jobs=4)]: Done 3248 tasks      | elapsed:  2.9min
[Parallel(n_jobs=4)]: Done 5840 tasks      | elapsed:  2.9min
[Parallel(n_jobs=4)]: Done 9008 tasks      | elapsed:  3.0min


Final speed_17d_arr shape: (9004, 17)
Saved joint-speed summary to: /content/drive/MyDrive/cs229/vitpose_sub8_train_joint_speeds.npz


[Parallel(n_jobs=4)]: Done 9537 out of 9537 | elapsed:  3.1min finished
